# EfficientNet with Test-Time Augmentation (TTA)

This notebook implements Test-Time Augmentation:
- Multiple augmented versions of test images
- Average predictions across all versions
- Improves robustness and accuracy

TTA variations:
- Original image
- Horizontal flip
- Vertical flip
- Transpose
- Rotations (90, 180, 270)

In [11]:
from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import albumentations as Albu
import pandas as pd
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
from tqdm import tqdm
import os
import sys
sys.path.append('../../..')
from utils.dataset import PandasDataset
from utils.metrics import model_checkpoint, evaluation, format_metrics, calculate_metrics
from utils.train import train_model
from utils.models import EfficientNetApi

In [12]:
seed = 42
batch_size = 6
num_workers = 4
output_classes = 5
init_lr = 3e-4
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

ROOT_DIR = '../../..'
data_dir = '../../../../dataset'
images_dir = os.path.join(data_dir, 'tiles')

Using device: cuda


In [13]:
load_model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes, dropout_rate=0.6)
model = model.to(device)

## Load Dataset

In [14]:
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
df_train_.columns = df_train_.columns.str.strip()
train_indexes = np.where((df_train_['fold'] != 3))[0]
valid_indexes = np.where((df_train_['fold'] == 3))[0]

df_train = df_train_.loc[train_indexes]
df_val = df_train_.loc[valid_indexes]
df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")

transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
])

train_dataset = PandasDataset(images_dir, df_train, transforms=transforms)
valid_dataset = PandasDataset(images_dir, df_val, transforms=None)
test_dataset = PandasDataset(images_dir, df_test, transforms=None)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(train_dataset)
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(valid_dataset)
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(test_dataset)
)

## Training (Standard)

In [15]:
optimizer = optim.Adam(model.parameters(), lr=init_lr/warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier=warmup_factor, total_epoch=warmup_epochs, after_scheduler=scheduler_cosine)

train_model(
    model=model,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    checkpoint=model_checkpoint,
    device=device,
    loss_function=loss_function,
    path_to_save_metrics="logs/tta-base.txt",
    path_to_save_model="models/tta-base.pth",
    patience=5,
)

Epoch 1/50



100%|██████████| 301/301 [01:36<00:00,  3.13it/s]


VAL_LOSS     0.286
VAL_ACC      Mean: 50.204 | Std: 1.203 | 95% CI: [48.255, 52.133]
VAL_KAPPA    Mean: 0.794 | Std: 0.010 | 95% CI: [0.777, 0.811]
VAL_F1       Mean: 0.454 | Std: 0.012 | 95% CI: [0.435, 0.474]
VAL_RECALL   Mean: 0.462 | Std: 0.012 | 95% CI: [0.444, 0.482]
VAL_PRECISION Mean: 0.517 | Std: 0.013 | 95% CI: [0.495, 0.538]
Salvando o melhor modelo... 0.0 -> 0.7944781276880124
Epoch 2/50



100%|██████████| 301/301 [01:37<00:00,  3.08it/s]
/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1087: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)


VAL_LOSS     0.313
VAL_ACC      Mean: 51.361 | Std: 1.179 | 95% CI: [49.363, 53.296]
VAL_KAPPA    Mean: 0.737 | Std: 0.012 | 95% CI: [0.717, 0.756]
VAL_F1       Mean: 0.406 | Std: 0.011 | 95% CI: [0.388, 0.426]
VAL_RECALL   Mean: 0.415 | Std: 0.010 | 95% CI: [0.399, 0.432]
VAL_PRECISION Mean: 0.528 | Std: 0.015 | 95% CI: [0.503, 0.552]
Epoch 3/50



...

VAL_LOSS     0.420
VAL_ACC      Mean: 52.083 | Std: 1.217 | 95% CI: [50.139, 54.186]
VAL_KAPPA    Mean: 0.725 | Std: 0.014 | 95% CI: [0.702, 0.748]
VAL_F1       Mean: 0.403 | Std: 0.012 | 95% CI: [0.384, 0.422]
VAL_RECALL   Mean: 0.415 | Std: 0.010 | 95% CI: [0.398, 0.433]
VAL_PRECISION Mean: 0.527 | Std: 0.010 | 95% CI: [0.511, 0.545]
Epoch 5/50



100%|██████████| 301/301 [01:29<00:00,  3.38it/s]


VAL_LOSS     0.327
VAL_ACC      Mean: 57.844 | Std: 1.144 | 95% CI: [55.956, 59.723]
VAL_KAPPA    Mean: 0.782 | Std: 0.013 | 95% CI: [0.759, 0.802]
VAL_F1       Mean: 0.520 | Std: 0.012 | 95% CI: [0.500, 0.540]
VAL_RECALL   Mean: 0.527 | Std: 0.012 | 95% CI: [0.508, 0.545]
VAL_PRECISION Mean: 0.586 | Std: 0.013 | 95% CI: [0.564, 0.606]
Epoch 6/50



100%|██████████| 301/301 [01:29<00:00,  3.37it/s]


VAL_LOSS     0.382
VAL_ACC      Mean: 57.753 | Std: 1.180 | 95% CI: [55.845, 59.778]
VAL_KAPPA    Mean: 0.787 | Std: 0.012 | 95% CI: [0.766, 0.807]
VAL_F1       Mean: 0.492 | Std: 0.012 | 95% CI: [0.471, 0.513]
VAL_RECALL   Mean: 0.490 | Std: 0.011 | 95% CI: [0.471, 0.509]
VAL_PRECISION Mean: 0.568 | Std: 0.013 | 95% CI: [0.545, 0.589]

Early stopping at epoch 6. No improvement for 5 epochs.
Best epoch: 1 with kappa: 0.7945


## Test-Time Augmentation Implementation

In [16]:
def tta_predict(model, image, device):
    """Apply test-time augmentation and return averaged predictions"""
    model.eval()
    predictions = []
    
    # Original
    with torch.no_grad():
        pred = torch.sigmoid(model(image))
        predictions.append(pred)
    
    # Horizontal flip
    with torch.no_grad():
        flipped = torch.flip(image, dims=[3])
        pred = torch.sigmoid(model(flipped))
        predictions.append(pred)
    
    # Vertical flip
    with torch.no_grad():
        flipped = torch.flip(image, dims=[2])
        pred = torch.sigmoid(model(flipped))
        predictions.append(pred)
    
    # Horizontal + Vertical flip
    with torch.no_grad():
        flipped = torch.flip(image, dims=[2, 3])
        pred = torch.sigmoid(model(flipped))
        predictions.append(pred)
    
    # Transpose
    with torch.no_grad():
        transposed = torch.transpose(image, 2, 3)
        pred = torch.sigmoid(model(transposed))
        predictions.append(pred)
    
    # Average all predictions
    return torch.stack(predictions).mean(dim=0)

def evaluate_with_tta(model, dataloader, device):
    """Evaluate model with TTA"""
    model.eval()
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets, _ in tqdm(dataloader, desc="TTA Evaluation"):
            inputs = inputs.to(device)
            
            # Apply TTA
            predictions = tta_predict(model, inputs, device)
            
            all_predictions.append(predictions.cpu())
            all_targets.append(targets)
    
    all_predictions = torch.cat(all_predictions, dim=0).numpy()
    all_targets = torch.cat(all_targets, dim=0).numpy()
    
    # Compute metrics
    metrics = format_metrics(calculate_metrics(all_predictions, all_targets))
    return metrics

print("TTA functions defined")

TTA functions defined


## Test WITHOUT TTA (Baseline)

In [17]:
model.load_state_dict(torch.load("models/tta-base.pth"))
response = evaluation(model, test_loader, device)
result_no_tta = format_metrics(response[0])
print("\n=== TEST RESULTS WITHOUT TTA ===")
print(result_no_tta)

100%|██████████| 266/266 [01:09<00:00,  3.83it/s]



=== TEST RESULTS WITHOUT TTA ===
VAL_ACC      Mean: 47.515 | Std: 1.224 | 95% CI: [45.540, 49.560]
VAL_KAPPA    Mean: 0.781 | Std: 0.013 | 95% CI: [0.760, 0.801]
VAL_F1       Mean: 0.428 | Std: 0.012 | 95% CI: [0.408, 0.450]
VAL_RECALL   Mean: 0.433 | Std: 0.012 | 95% CI: [0.414, 0.454]
VAL_PRECISION Mean: 0.505 | Std: 0.014 | 95% CI: [0.482, 0.528]


## Test WITH TTA

In [18]:
model.load_state_dict(torch.load("models/tta-base.pth"))
metrics_tta = evaluation(model, test_loader, device)
result_tta = format_metrics(metrics_tta)
print("\n=== TEST RESULTS WITH TTA ===")
print(result_tta)

TTA Evaluation: 100%|██████████| 266/266 [07:29<00:00,  1.69s/it]


ValueError: Classification metrics can't handle a mix of continuous-multioutput and multilabel-indicator targets

## Comparison

In [ ]:
print("\n" + "="*60)
print("COMPARISON: TTA vs No TTA")
print("="*60)
print("\nWITHOUT TTA:")
print(result_no_tta)
print("\nWITH TTA:")
print(result_tta)
print("="*60)